<img src="https://s0.cptec.inpe.br/webcptec/sites/www/assets/img/logo_cptec.png" align="right" width="64"/>

# <div style="text-align: center;"><span style="color:#336699; font-size: 1.2em;">MERGE<br><span style="color:#336699; font-style: italic;">      Extração de séries temporais de precipitação acumulada diária.</span></span></div>
<hr style="border:2px solid #0077b9;">

<br/>

<div style="text-align: center;font-size: 90%;">
    Katiusca Briones Estébanez<sup><a href="https://orcid.org/0000-0003-3425-9128"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    Douglas Uba<sup><a href=""><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    Alex de Almeida Fernandes<sup><a href="https://orcid.org/0000-0003-1520-5896"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    <br/><br/>
    Divisão de Previsão de Tempo e Clima, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Rodovia Presidente Dutra, km 40, Cachoeira Paulista, SP 12630-000, Brazil
    <br/><br/>
    Contato: <a href="mailto:douglas.uba@inpe.br">douglas.uba@inpe.br</a>
    <br/><br/>
    Ultíma Atualização: 06 de Março de 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 15%; margin-right: 15%;">
<b>Obetivo.</b> O objetivo deste Jupyter Notebook é extrair séries temporais de precipitação de localidades específicas definidas pelo usuário, geradas a partir dos dados acumulados diários do produto MERGE do INPE. O dado MERGE consiste na combinação dos dados de superfície de estações pluviométricas, em conformidade com o padrão da Organização Meteorológica Mundial, e dados de estimativas de precipitação por satélite IMERG/GPM. Esta combinação torna a estimativa de satélite mais precisa e permite o uso em locais onde não há observações de superfície.

<br/>
<div style="text-align: justify;">
    Para a obtenção das séries temporais neste notebook, é utilizado o serviço STAC na linguagem Python para descoberta e acesso aos produtos de dados de sensoriamento remoto disponíveis no catálogo do INPE. Os arquivos, em formato GRIB2, são acessados e manipulados para visualizar espacialmente a distribuição dos dados acumulados diários de precipitação para o período de tempo selecionado, assim como para a extração das séries temporais. Este notebook possibilita ao usuário modificar no código as coordenadas da localidade da qual as séries temporais serão extraídas.

<br/>
<div style="text-align: justify;">    
    Para fins de comprovação da similaridade da série do MERGE com dados observados, é fornecido o código para comparar visualmente a série obtida com arquivos de precipitação observada do INMET (formato CSV) disponíveis no site <a href="https://mapas.inmet.gov.br/">https://mapas.inmet.gov.br</a>. Gráficos de linha e de dispersão são utilizados para a comparação visual correspondente.
<br/><br/>


## 1. Serviço STAC e Coleções

Boa parte dos produtos de imagem disponibilizados no catálogo de imagens do INPE são disponibilizados de maneira aberta na forma de arquivos otimizados para cloud, o denominado formato Cloud Optimized GeoTIFF (COG). Este formato permite que as aplicações possam utilizar as imagens através da Web com o melhor compromisso possível, incluindo o uso de pirâmide de multi-resolução para aplicações de visualização ou até mesmo a recuperação parcial de porções de uma imagem. O COG e o serviço de análise da BIG/INPE permitem várias análises e facilidades. 

Os produtos de dados podem ser consultados utilizando uma interface de programação de aplicações baseada no padrão aberto SpatioTemporal Asset Catalog (STAC). Alguns conceitos são importantes para entender o acesso aos dados com este padrão:

- **Catalog**: É um tipo de objeto que fornece uma estrutura para vincular vários itens ou coleções STAC juntos ou mesmo outros catálogos. Na figura acima, o catálogo é composto de três coleções: Landsat/OLI, CBERS4/WFI e Sentinel-2/MSI.

- **Collection:** É uma especialização do catálogo que permite incluir informações adicionais sobre uma determinada coleção espaço-temporal. Uma coleção pode conter informações como o conjunto de bandas espectrais disponíveis das imagens, a extensão geográfica ou área de cobertura das imagens, o período de tempo que compreende a coleção, entre outras informações. Em geral, através da coleção chegamos aos itens dessa coleção.

- **Item**: Corresponde à unidade atômica de metadados, fornecendo *links* para os *assets* associados. Um *Item* é descrito através da notação GeoJSON, como uma feição (*feature*) contendo atributos específicos como a coleção a que ele pertence, propriedades temporais, *links* para os *assets* e coleções ou catálogos associados. 

- **Asset**: Um *asset* é qualquer recurso geoespacial, como um arquivo de imagem ou arquivo vetorial, contendo informações sobre a superfície da Terra, em um determinado espaço e tempo.


A especificação conceitual do STAC permite dois tipos de implementações:

- **STAC estático:** Baseado em um conjunto de documentos JSON ligados que podem ser facilmente navegados. Ex: [CBERS na AWS](https://cbers-stac-1-0-0.s3.amazonaws.com/CBERS4/catalog.json).

- **STAC dinâmico:** Baseado em uma API RESTful, de modo que a navegação é realizada através de uma API de serviço web que permite realizar consultas utilizando uma linguagem padrão para acessar subconjuntos do catálogo. Ex: [BDC-STAC](https://data.inpe.br/bdc/stac/v1).


<br/>
<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;font-size: 75%; border-style: solid; border-color: #0077b9; border-width: 1px; padding: 5px;">
    <b>Nota:</b> Como parte do aperfeiçoamento dos produtos e serviços disponibilizados pelo INPE à sociedade, encontra-se em desenvolvimento o novo portal <a href="https://data.inpe.br/">https://data.inpe.br/</a>, que faz parte da modernização da infraestrutura de serviços para acesso às imagens de satélites do acervo do instituto. Esse portal foi criado com o intuito de facilitar a pesquisa e obtenção das imagens disponibilizadas gratuitamente. Esse novo serviço tem como base as tecnologias desenvolvidas no projeto Brazil Data Cube e está ancorado dentro do Programa Base de Informações Georreferenciadas (BIG) do INPE. Para navegar pelas coleções disponibilizadas no serviço STAC do INPE, utilize a instância do [STAC Browser](https://data.inpe.br/stac/browser/).
</div>


#### Acessando o servico SpatioTemporal Asset Catalog (STAC do INPE)

In [ ]:
import pystac_client
service = pystac_client.Client.open('https://data.inpe.br/bdc/stac/v1/')
service

### 2. Leitura dos itens da coleção e gráfico espacial.

#### Função para baixar e ler itens da coleção de precipitação acumulada diária do MERGE.

Criamos a função _Download_and_read_merge_stac_ para fazer download dos itens da coleção do _prec_merge_daily-1_ que contêm os dados acumulados diários de precipitação. 
A leitura dos dados é feita com o pacote _Xarray_, que fornece estruturas de dados para trabalhar com arrays e conjuntos de dados multidimensionais rotulados. Ele amplia as capacidades do _NumPy_ e do _Pandas_, facilitando o gerenciamento e a análise de dados que possuem dimensões (como tempo, latitude e longitude) e metadados associados.

Os dados de precipitação do MERGE estão em formato GRIB2 e não podem ser acessados diretamente pelo _XARRAY_, por tanto, vamos fazer o download e depois a leitura. Usamos a biblioteca _requests_ para fazer o download do link HTTPS. Com os arquivos armazenados localmente, usamos _open_mfdataset_ da XARRAY. 

Para a função, deve-se passar o serviço do _pystac_client_ que foi aberto, uma data inicial e final e o diretório onde os dados serão baixados. Se a data inicial for igual à data final, apenas um dia será baixado.

In [ ]:
from dateutil.parser import parse
from pathlib import Path
import pystac
import requests
import xarray as xr

def download_and_read_merge_stac(
    stac_service: str,
    start_date: str,
    end_date: str,
    output_dir: str = "merge_data"
):
    """
    Busca, baixa e lê dados MERGE em formato GRIB2 do catálogo STAC do INPE,
    filtrando por um período de datas. 

    Args:
        stac_catalog_url (str): URL do catálogo/item STAC do INPE que contém o MERGE diário.
        start_date (str): Data inicial (formato ISO ou legível, ex: '2024-01-01').
        end_date (str): Data final (formato ISO ou legível, ex: '2024-01-31').
        output_dir (str): Pasta onde salvar os arquivos baixados.

    Returns:
        ds (xarray.Dataset): Dataset final contendo os dados do merge com a dimensão tempo.
    """
    # Converte datas para objetos datetime
    start_dt = parse(start_date)
    end_dt = parse(end_date)

    # Cria diretório de saída se não existir
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    downloaded_files = []

    if isinstance(service, pystac.Catalog) and stac_service.id == 'INPE':  # acesso stac server do INPE
        item_search = service.search(datetime=start_date+'/'+end_date,     # Pesquisa apenas as datas de interesse na coleção prec_merge_daily-1
                             collections=['prec_merge_daily-1'])
        for asset in item_search.items():                                  # verifica os assets dos itens encontrados anteriormente
            if asset.assets['merge_daily'].href.endswith(".grib2"):        # Acessa apenas os links dos dados GRIB2, há outros como o idx e ctl.
                file_url = asset.assets['merge_daily'].href
                filename = Path(file_url).name
                file_path = output_path / filename

                print(f"Baixando: {file_url}")
                response = requests.get(file_url, stream=True)             # Baixa os dados 
                response.raise_for_status()

                with open(file_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

                        print(f"Arquivo salvo: {file_path}")
                        downloaded_files.append(file_path) # Adiciona novo arquivo baixado na lista de arquivos para abertura com o xarray
    else:
        raise ValueError("STAC URL deve apontar para um Catálogo ou Item.")

    # Verifica se algum arquivo foi baixado
    if not downloaded_files:
        raise FileNotFoundError("Nenhum arquivo .grib2 foi encontrado no período especificado.")

    # Lê os arquivos com xarray
    print("Lendo arquivos com xarray...")
          
    # Lê múltiplos arquivos com open_mfdataset, aplicando a função preprocess
    ds = xr.open_mfdataset(
        downloaded_files,
        engine='cfgrib',
        combine='nested',
        concat_dim='time',
        decode_timedelta=False,
        backend_kwargs={"indexpath": ""}  # Desliga a indexação em disco
    )

    return ds

Usando a função _download_and_read_merge_stac_, selecionamos a data de início e fim para o qual os dados vão ser acessados. Adicionalmente, selecionamos a pasta remota (no BDC_Lab) onde serão armazenados tais arquivos.

In [ ]:
ds = download_and_read_merge_stac(service, '2026-01-01', '2026-01-30', './data/merge')
ds = ds.sortby('time')  # O download pode não ter ocorrido na ordem de datas, portanto ordena o dado pela dimensão _time_

#Note-se que os dados de longitude nos arquivos baixados estão no formato de 0°-360° E.
ds

Fazemos a correção dos dados de longitude para o formato 180°W  -180°E.

In [ ]:
ds = ds.assign_coords(longitude=((ds.longitude+180) % 360)-180)
ds

#### Função para visualizar os itens.

Criamos a função _plot_merge_grid_ para visualizar um arranjo que apresenta a distribuição espacial da precipitação para cada item. Cada item corresponde a um dia de precipitação acumulada.

In [ ]:
import os
import pyproj

# Automatically find the correct PROJ data directory and set the environment variable
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd

def plot_merge_grid(dataset, variable_name='rdp'):
    """
    Cria um arranjo de n mapas baseados na dimensão tempo do dataset MERGE.
    """
    # Seleciona os primeiros 31 tempos disponíveis
    times = dataset.time.values#[:31]
    num_plots = len(dataset.time.values)
    
    cols = 6
    rows = int(np.ceil(num_plots / cols))
    
    fig, axes = plt.subplots(
        rows, cols, 
        figsize=(20, 4 * rows), 
        subplot_kw={'projection': ccrs.PlateCarree()},
        constrained_layout=True
    )
    
    axes_flat = axes.flatten()
    
    # Extrai o nome amigável da variável (ex: Precipitation)
    long_name = dataset[variable_name].attrs.get('long_name', variable_name)

    print(f"Gerando arranjo de {num_plots} mapas para: {long_name}")

    for i, t in enumerate(times):
        ax = axes_flat[i]
        data_slice = dataset[variable_name].sel(time=t)
        
        # Plotagem espacial
        im = data_slice.plot(
            ax=ax, 
            transform=ccrs.PlateCarree(),
            add_colorbar=False,
            cmap='YlGnBu',
            vmin=0, vmax=50 # Ajuste de escala para chuva diária
        )
        
        # Adiciona detalhes geográficos
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        
        # Título com a data
        data_str = pd.to_datetime(t).strftime('%d/%m/%Y')
        ax.set_title(data_str, fontsize=18)

    # Remove eixos excedentes
    for j in range(i + 1, len(axes_flat)):
        fig.delaxes(axes_flat[j])

    # Adiciona uma barra de cores global
    cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.02, pad=0.02)
    cbar.set_label(f'{long_name} (mm/dia)')

    plt.suptitle(f"Monitoramento de Precipitação MERGE/INPE - {len(times)} dias", fontsize=20, y=1.05)
    
    # Exibe o gráfico na tela em vez de salvar
    print("Exibindo arranjo de mapas...")
    plt.show()

Visualizamos o arrango dos itens (arquivos diários de precipitação acumulada) segundo as datas selecionadas.

In [ ]:
# Gera o arranjo visual
plot_merge_grid(ds)

### 3. Extração da série temporal de precipitação acumulada diária do MERGE.

Um método que facilita em grande medida o acesso aos cubos de dados no *XARRAY* é o _.sel()_, que é a ferramenta principal para indexação e seleção de dados com base em rótulos (labels), permitindo acessar dados por meio de coordenadas (ex.: latitude, tempo) em vez de posições inteiras.

Assim, simplesmente, enviamos como argumentos do método a latitude e longitude para as quais vamos a obter a série temporal. O _method = "nearest"_ pede ao _.sel()_ selecionar o ponto mais próximo identificado nos dados do MERGE.

In [ ]:
MERGE_Guarulhos = ds.sel(latitude=-23.44, longitude=-46.47, method='nearest')
MERGE_Guarulhos

Selecionamos a variável da precipitação _'prec'_ na série obtida pelo _.sel()_. 

Adicionalmente convertemos o xarray obtido a uma série Pandas e normalizamos o índice (data) para padronizar as datas da série do MERGE. Nos próximos passos vamos padronizar também as datas da série dos dados observados do INMET.

Assim, obtemos a série temporal da precipitação para o ponto geográfico selecionado.

In [ ]:
#Selecionar a variável de precipitação "rpd" ou "prec"
MERGE_precip_Guarulhos = MERGE_Guarulhos.rdp
#MERGE_precip_Guarulhos = MERGE_Guarulhos.prec

# Convert the xarray DataArray to a pandas Series
MERGE_precip_Guarulhos = MERGE_precip_Guarulhos.to_series()
    
# Normalize the INPE index just to be safe
MERGE_precip_Guarulhos.index = MERGE_precip_Guarulhos.index.normalize()
MERGE_precip_Guarulhos

### 4. Criação de série temporal de precipitação do INMET.

#### Função para ler o arquivo .CSV e obter a série de precipitação do INMET.

Vamos comparar os dados do MERGE com os dados observados do INMET. 

A função _get_precipitation_ts_ lê um arquivo .CSV obtido do site https://mapas.inmet.gov.br/ para identificar e selecionar a série temporal da precipitação observada do INMET. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def get_precipitation_INMET(path):
    """
    #Lê um arquivo CSV gerado pelo INMET (https://mapas.inmet.gov.br/) e gera uma série temporal de precipitação 
    (ÚLTIMA coluna de dados), usando as colunas de data/hora para o índice.
    """

    file_path = path

    print(f"Carregando arquivo: {file_path}")
    
    # 1. Carregamento robusto
    try:
        # Lê o CSV sem cabeçalho fixo primeiro para inspecionar ou lê normal
        # Assumindo o formato padrão do arquivo enviado: separador ';' e decimal ','
        df = pd.read_csv(
            file_path,
            sep=';',
            decimal=',',
            quotechar='"',
            encoding='utf-8',
            on_bad_lines='skip')
    except Exception as e:
        print(f"Erro crítico ao ler CSV: {e}")
        #return None

    # 2. Leitura de Data e Hora (Índice)
    # As datas estão na primeira coluna. Para evitar datas duplicadas (ex: várias medições no mesmo dia),
    # precisamos combinar com a hora (coluna 2) se ela existir.
    
    try:
        col_data = df.columns[0] # Primeira coluna (Datas)
        col_hora = df.columns[1] # Segunda coluna (Horas)
        
       # Tratamento da hora: garante 4 dígitos (ex: 0 -> '0000', 300 -> '0300')
        raw_time = df[col_hora].fillna(0).astype(int).astype(str).str.zfill(4)
        raw_date = df[col_data]
        
        # Criação do índice Datetime
        df.index = pd.to_datetime(
        raw_date + ' ' + raw_time, 
        format='%d/%m/%Y %H%M', 
        #format='%Y/%m/%d %H%M', 
        errors='coerce'
        )

        # Remove linhas onde a data não pôde ser convertida
        df = df[df.index.notnull()]
        df = df.sort_index()
        df.index.name = 'DataHora_UTC'

    except Exception as e:
        print(f"Erro ao processar as datas/horas: {e}")
        #return None

    # 3. Seleção da Última Coluna
    # iloc[:, -1] seleciona a última coluna do DataFrame
    last_col_name = df.columns[-1]
    print(f"\nFocando na última coluna identificada: '{last_col_name}'")
   
    # Extrai a série e remove vazios (NaN)
    precip_ts = df.iloc[:, -1].dropna()
    precip_ts.index = precip_ts.index.normalize()
   
    # 4. Exibição dos Resultados
    print("-" * 50)
    if precip_ts.empty:
        print(f"A série '{last_col_name}' não possui dados válidos (está vazia).")
    else:
        print(f"Série Temporal Gerada: {last_col_name}")
        print(f"Total de registros: {len(precip_ts)}")
        print(f"Início: {precip_ts.index.min()}")
        print(f"Fim:    {precip_ts.index.max()}")
        
    # Estatísticas básicas se for numérico
    try:
        print(f"Média:  {precip_ts.mean():.2f}")
        print(f"Mínimo: {precip_ts.min()}")
        print(f"Máximo: {precip_ts.max()}")
    except:
        print("Dados não numéricos (estatísticas ignoradas).")
            
    print("\nSérie completa:Primeiros 5 registros:")
    #print(precip_ts.head())
    print(precip_ts[:])
        
    print("-" * 50)

    return precip_ts

Obtemos a série temporal de precipitação diária do INMET.

In [ ]:
INMET_precip_Guarulhos = get_precipitation_INMET('tabela_chuva_diaria.csv')

### 5. Comparação visual das séries do MERGE e do INMET.

#### Função para criar gráficos de linha das séries temporais do MERGE e do INMET.

A função _plot_ts_ cria um gráfico para comparar as séries temporais do MERGE e do INMET.

In [ ]:
import matplotlib.dates as mdates

def plot_ts(serie_MERGE, serie_INMET, station):
    try:
        plt.figure(figsize=(8, 4))
    
        # Plot INPE Merge (as a line or bar, depending on preference)
        if not MERGE_precip_Guarulhos.empty:
            plt.bar(serie_MERGE.index, serie_MERGE.values, label='INPE Merge', color='orange', linewidth=1.5)

        plt.bar(serie_INMET.index, serie_INMET.values, label='Estação INMET', color='cyan', linewidth=1.5)

    except Exception as e:
        print(f"Erro crítico ao ler as séries temporal: {e}")
        
    # Graph formatting
    plt.title(f'Precipitação {station}: MERGE vs. Estação INMET')
    plt.xlabel('Data', fontsize=9)
    plt.ylabel('Precipitação Diária (mm)', fontsize=9)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=8)

    # Format the x-axis to show dates nicely
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().tick_params(axis='x', labelsize=6)
    plt.gca().tick_params(axis='y', labelsize=6)
    plt.gcf().autofmt_xdate() # Rotates dates so they don't overlap

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd


def plot_ts(serie_MERGE, serie_INMET, station):
    try:
        #plt.figure(figsize=(10, 4.5))
        plt.figure(figsize=(6, 4.5))

        # Definição da largura da barra (em dias) e deslocamento lateral
        width = 0.35
        offset = pd.Timedelta(days=width / 2)

        # Plot INPE Merge (deslocado para a esquerda)
        if serie_MERGE is not None and not serie_MERGE.empty:
            plt.bar(
                serie_MERGE.index - offset,
                serie_MERGE.values,
                width=width,
                label="INPE Merge",
                color="blue",
                align="center",
            )

        # Plot Estação INMET (deslocado para a direita)
        if serie_INMET is not None and not serie_INMET.empty:
            plt.bar(
                serie_INMET.index + offset,
                serie_INMET.values,
                width=width,
                label="Estação INMET",
                color="cyan",
                align="center",
            )

    except Exception as e:
        print(f"Erro crítico ao ler as séries temporais: {e}")

    # Formatação do gráfico
    plt.title(f"Precipitação {station}: MERGE vs. Estação INMET", fontsize=18)
    plt.xlabel("Data", fontsize=10)
    plt.ylabel("Precipitação Diária (mm)", fontsize=10)
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.legend(fontsize=10)

    # Formatação do eixo X para datas
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    plt.gca().tick_params(axis="x", labelsize=7)
    plt.gca().tick_params(axis="y", labelsize=7)
    plt.gcf().autofmt_xdate()

    plt.tight_layout()
    plt.show()

#### Função para criar um gráfico de dispersão das séries temporais do MERGE e do INMET.

A função _plot_ts_scatter cria um gráfico de dispersão para visualizar o grau de dispersão das séries temporais do MERGE e do INMET.

In [ ]:
import matplotlib.dates as mdates

def plot_ts_scatter(serie_MERGE, serie_INMET, station):
    try:
        #plt.figure(figsize=(12, 6))
        fig, ax = plt.subplots()

        # Plot INPE Merge (as a line or bar, depending on preference)
        if not MERGE_precip_Guarulhos.empty:
            ax.scatter(serie_MERGE, serie_INMET, label='INPE Merge', color='blue', linewidth=1.5)
    
            # Add an identity line (y=x) passing through the origin with a slope of 1
            ax.axline((0, 0), slope=1, color='black', linestyle='--', label='y=x line')

            #Set equal limits for x and y axes for a true 45-degree appearance
            current_limits = [min(ax.get_xlim()[0], ax.get_ylim()[0]), 
                               max(ax.get_xlim()[1], ax.get_ylim()[1])]
            ax.set_xlim(current_limits)
            ax.set_ylim(current_limits)

    except Exception as e:
        print(f"Erro crítico ao ler as séries temporal: {e}")
        
    # Graph formatting
    plt.title(f'Precipitação {station}: MERGE vs. Estação INMET')
    plt.xlabel('Precipitação Diária (mm)', fontsize=9)
    plt.ylabel('Precipitação Diária (mm)', fontsize=9)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=8)
    plt.tick_params(axis='x', labelsize=6)
    plt.tick_params(axis='y', labelsize=6)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_ts(MERGE_precip_Guarulhos, INMET_precip_Guarulhos, 'Guarulhos')

In [ ]:
plot_ts_scatter(MERGE_precip_Guarulhos, INMET_precip_Guarulhos, 'Guarulhos')

#### Visualizando um outro ponto geográfico.

Com as funções criadas, podemos modificar fácilmente o código para selecionar qualquer coordenada geográfica e gerar as séries temporais e os gráficos correspondentes.

Vamos visualizar os dados da localidade do aeroporto de Campo de Marte, em São Paulo.

In [ ]:
MERGE_CM = ds.sel(latitude=-23.51, longitude=-46.63, method='nearest')

#Selecionar a variável de precipitação "rpd" ou "prec"
MERGE_precip_CM = MERGE_CM.rdp
#MERGE_precip_CM = MERGE_CM.prec

# Convert the xarray DataArray to a pandas Series
MERGE_precip_CM = MERGE_precip_CM.to_series()
    
# Normalize the INPE index just to be safe
MERGE_precip_CM.index = MERGE_precip_CM.index.normalize()
MERGE_precip_CM

In [ ]:
INMET_precip_CM = get_precipitation_INMET (path='tabela_chuva_diaria_campo_marte.csv')

In [ ]:
plot_ts(MERGE_precip_CM, INMET_precip_CM, 'Campo de Marte')

In [ ]:
plot_ts_scatter(MERGE_precip_Guarulhos, INMET_precip_Guarulhos, 'Guarulhos')